# Tool Calling with LangChain and Local Ollama models

This notebook demonstrates the best practice for defining tools and using them with the **gpt-oss:20b** model via Ollama. I am using langchain here, as I plan to use LangChain and LangGraph for Agents demos in next notebooks.

### Prerequisites
Ensure you have Ollama installed and the model pulled:
```bash
pip install -U langchain-ollama langchain-core pydantic

ollama serve
ollama pull gpt-oss:20b
```
_Note: You can use any other model depending on the resources you have._ 

### Tool Definition - Using "@tool" Decorator (Good for Simplicity)

This is the most common and "Pythonic" way. LangChain automatically parses the function name, its docstring, and its type hints to create the tool's schema.

Key Rule: Your docstring is the "manual" the LLM reads. Be extremely descriptive about when to use the tool and what the arguments represent.

In [31]:
from langchain_core.tools import tool

@tool
def get_weather(location: str, unit: str = "celsius"):
    """
    Consult this tool to get the current weather for a specific location.

    Args:
        location: The city and state, e.g. San Francisco, CA
        unit: The temperature unit (celsius or fahrenheit)
    """
    # Your logic goes here (e.g., calling an actual API)
    return f"The weather in {location} is 22 degrees {unit}."

### Decorator + Pydantic (Good for Production code)
For complex tools with multiple arguments or strict validation needs, define a Pydantic model for the input. This prevents the LLM from passing "hallucinated" or incorrectly formatted parameters.

- Pydantic Schemas: By defining WeatherInput, you give the LLM a strict specification. This prevents the model from guessing variable names or passing the wrong data types.
- Explicit Docstrings: In LangChain, the tool's docstring is literally injected into the prompt. Writing clear, instructional docstrings is the most effective way to "guide" the agent.
- bind_tools Integration: This is the native way LangChain interacts with tool-calling models. It handles the conversion of your Python function into the JSON format that gpt-oss:20b expects.

In [32]:
from typing import Optional
from pydantic import BaseModel, Field
from langchain_core.tools import tool
from langchain_ollama import ChatOllama

# --- STEP 1: Define the Input Schema ---
# This ensures the LLM knows exactly what parameters to send.
class WeatherInput(BaseModel):
    location: str = Field(description="The city and state, e.g. San Francisco, CA")
    unit: str = Field(default="celsius", description="The temperature unit (celsius or fahrenheit)")

# --- STEP 2: Create the Tool ---
# The docstring acts as the 'instructions' for the AI.
@tool("get_current_weather", args_schema=WeatherInput)
def get_weather(location: str, unit: str = "celsius"):
    """Consult this tool to get the current weather for a specific location."""
    # In a real app, you'd call an API here.
    return f"The weather in {location} is 22 degrees {unit}."

tools = [get_weather]
print(f"Tool '{get_weather.name}' defined successfully.")

Tool 'get_current_weather' defined successfully.


### Initialize the Model and Bind Tools
We use `ChatOllama` and the `.bind_tools()` method to tell the model which tools it can use.



In [33]:
# Initialize model (Make sure Ollama is running!)
llm = ChatOllama(
    model="gpt-oss:20b",
    temperature=0.3,
    format="json" # Some models perform better in JSON mode for tools (constrained decoding)
)

# Bind the tools to the LLM
llm_with_tools = llm.bind_tools(tools)

# Ask a question that requires a tool
query = "What is the weather like in New York?"
response = llm_with_tools.invoke(query)

print("Model Response Message:")
print(response)

Model Response Message:
content='' additional_kwargs={} response_metadata={'model': 'gpt-oss:20b', 'created_at': '2026-01-03T23:46:41.041553Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1350826542, 'load_duration': 146982917, 'prompt_eval_count': 169, 'prompt_eval_duration': 305416334, 'eval_count': 53, 'eval_duration': 850270045, 'logprobs': None, 'model_name': 'gpt-oss:20b', 'model_provider': 'ollama'} id='lc_run--019b8641-85c9-7951-990f-6fea30076656-0' tool_calls=[{'name': 'get_current_weather', 'args': {'location': 'New York', 'unit': 'celsius'}, 'id': '719a58b5-89f0-4179-ba83-f7e78ff628b5', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 169, 'output_tokens': 53, 'total_tokens': 222}


### Extracting and Calling the Tool
When a model decides to use a tool, it populates the `tool_calls` attribute of the response.

In [34]:
if response.tool_calls:
    for tool_call in response.tool_calls:
        print(f"\nAction: Calling tool '{tool_call['name']}'")
        print(f"Arguments: {tool_call['args']}")
        
        # Execute the tool function
        result = get_weather.invoke(tool_call['args'])
        print(f"Observation: {result}")
else:
    print("No tool call was generated. The model answered directly:")
    print(response.content)


Action: Calling tool 'get_current_weather'
Arguments: {'location': 'New York', 'unit': 'celsius'}
Observation: The weather in New York is 22 degrees celsius.


## Summary

- This notebook shows how to use **LangChain tools with local Ollama models** (e.g., `gpt-oss:20b`).
- We first defined a simple tool using the `@tool` decorator — great for quick and clean prototypes.
- Then we created a **Pydantic-based tool schema** to strictly control input structure and prevent invalid tool calls.
- Clear **docstrings act as instructions for the LLM**, so being explicit improves reliability.
- We used `bind_tools()` so the model knows which tools are available and how to call them.
- When the LLM decides to use a tool, the details appear in the `tool_calls` field — which we then executed programmatically.
- Key takeaway: **Pydantic schemas + good docstrings = safer, more predictable tool calling**.
- This setup forms the foundation for future notebooks on **agents and LangGraph workflows**.